In [1]:
"""
main.py

End-to-end orchestration: data -> features -> walk-forward model ->
backtest. Run this after data_prep.py has pulled CRSP prices and
point-in-time S&P 500 membership from WRDS.

    python src/data_prep.py <wrds_username>   # one-time WRDS pull, caches to data/
    python src/main.py                        # runs the full research pipeline
"""

# for ensuring that the notebook imports the updated function files instead of using the one in cache
%load_ext autoreload
%autoreload 2

import pandas as pd
from pathlib import Path

from features import build_feature_panel
from model import run_walk_forward, summarize_ic
from backtest import compare_models
from visualize import plot_model_comparison, plot_ic_timeseries

DATA_DIR = Path.cwd().resolve().parent / "data"
OUTPUT_DIR = Path.cwd().resolve().parent / "output"

# Forward-return horizon in months. test_months/step_months are set equal to
# HORIZON below so each walk-forward test period's realized return window is
# disjoint from the next -- required for backtest.py to validly compound
# them as a sequential return series. See model.run_walk_forward's docstring.
HORIZON = 3



In [2]:

print("Loading CRSP price panel and point-in-time membership...")
daily_panel = pd.read_parquet(DATA_DIR / "prices_wrds.parquet")
membership = pd.read_parquet(DATA_DIR / "sp500_membership.parquet")


Loading CRSP price panel and point-in-time membership...


In [3]:
print("Building feature panel...")
feature_panel = build_feature_panel(daily_panel, membership, horizon=HORIZON)
print(f"Feature panel shape: {feature_panel.shape}")
print(f"Date range: {feature_panel['date'].min()} to {feature_panel['date'].max()}")
print(
    f"Unique PERMNOs represented (point-in-time members only): "
    f"{feature_panel['permno'].nunique()}"
)

OUTPUT_DIR.mkdir(exist_ok=True)
feature_panel.to_parquet(OUTPUT_DIR / "feature_panel.parquet", index=False)

Building feature panel...
Feature panel shape: (80685, 18)
Date range: 2012-01-31 00:00:00 to 2025-12-31 00:00:00
Unique PERMNOs represented (point-in-time members only): 751


In [4]:
feature_panel

,permno,cum_ret_index,mkt_cap,avg_dollar_vol,realized_vol,date,mom_1m,mom_3m,mom_12m_ex1,log_mkt_cap,log_dollar_vol,fwd_ret,mom_1m_z,mom_3m_z,mom_12m_ex1_z,realized_vol_z,log_mkt_cap_z,log_dollar_vol_z
12,10104,0.909792,1.417789e+11,9.977141e+08,0.009604,2012-01-31,0.102246,-0.137243,-0.194702,25.677534,20.720977,0.044340,0.581911,-1.795552,-0.805056,-0.816627,2.412160,2.272625
13,10104,0.943493,1.456606e+11,8.185006e+08,0.011857,2012-02-29,0.037043,-0.064759,-0.135835,25.704545,20.522985,-0.093324,-0.067986,-1.451836,-0.648039,-0.334194,2.408946,2.108568
14,10104,0.940430,1.450741e+11,1.099563e+09,0.012438,2012-03-31,-0.003247,0.139365,-0.118097,25.700510,20.818178,0.020626,-0.467092,0.096863,-0.719984,-0.188357,2.356913,2.465878
15,10104,0.950132,1.462681e+11,7.758613e+08,0.011946,2012-04-30,0.010317,0.044340,-0.184195,25.708707,20.469484,0.029320,0.305929,-0.129027,-1.005198,-0.599295,2.363086,2.093286
16,10104,0.855443,1.298354e+11,7.986041e+08,0.014791,2012-05-31,-0.099659,-0.093324,-0.133870,25.589533,20.498376,0.198148,-0.407942,-0.330610,-0.723287,-0.117092,2.295597,2.097098
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92051,93436,1.441167,1.076881e+12,2.522249e+10,0.022692,2025-08-31,0.083044,-0.036338,0.439781,27.705090,23.951002,0.288436,0.761595,-0.787535,1.340828,0.659644,2.908778,3.669326
92052,93436,1.919656,1.478249e+12,3.744436e+10,0.028232,2025-09-30,0.332015,0.399987,0.276119,28.021880,24.346122,0.011243,4.149459,2.490408,0.829468,1.803334,3.141362,3.389105
92053,93436,1.970763,1.518436e+12,3.879408e+10,0.032817,2025-10-31,0.026623,0.481038,0.779952,28.048702,24.381533,-0.057274,0.458562,2.911796,2.464783,1.551479,3.135075,3.497685
92054,93436,1.856851,1.430668e+12,3.582897e+10,0.033233,2025-11-30,-0.057801,0.288436,0.322753,27.989162,24.302023,-0.064301,-1.027582,1.747141,1.025374,1.768506,3.075283,3.553019


In [5]:
print("\nRunning walk-forward for the LightGBM ranker...")

# Returns a dataframe of out-of-sample predictions with columns: [date, permno, fwd_ret, pred]
# this is what backtest.py consumes
#preds = run_walk_forward(feature_panel, test_months=1, step_months=HORIZON, horizon=HORIZON)


Running walk-forward for the LightGBM ranker...


In [18]:
# Going into run_walk_forward()
import pandas as pd
import numpy as np
from lightgbm import LGBMRanker
from scipy.stats import spearmanr

from features import winsorize


FEATURE_COLS = [
    "mom_1m_z", "mom_3m_z", "mom_12m_ex1_z", "realized_vol_z",
    "log_mkt_cap_z", "log_dollar_vol_z",
]
TARGET_COL = "fwd_ret"

panel = feature_panel
train_months = 60
test_months = 1
step_months = 3
horizon = 3

results = []
embargo_months = max(0, horizon - 1)
print(f"Embargo Months: {embargo_months}")

Embargo Months: 2


In [7]:
def walk_forward_splits(
    dates: pd.Series,
    train_months: int = 60,
    test_months: int = 1,
    step_months: int = 1,
    embargo_months: int = 0,
):
    """
    Generator yielding (train_dates, test_dates) tuples.

    Expanding-window alternative: set train_months=None to use all history
    up to the test window each time, instead of a fixed lookback. A fixed
    rolling window (as implemented here) is often preferred because it
    keeps the model from being trained on a regime that's no longer
    representative (e.g. pre-2020 volatility structure).

    NOTE ON PURGING: with a 1-month-forward target, a training observation
    dated one month before the test window doesn't overlap the test
    period's information set (mom_12m_ex1's 12m lookback doesn't reach
    forward into the test month). That assumption breaks once the target
    itself looks further forward than 1 month: a training row's fwd_ret is
    only "known" `horizon` months after its feature date, so any training
    row within `horizon - 1` months of the test start has a label that
    wouldn't actually be realized yet at test time -- using it anyway is
    leakage. `embargo_months` drops exactly those trailing training months
    (run_walk_forward sets it to `horizon - 1` automatically). At
    embargo_months=0 (the 1-month-horizon default) this is a no-op and
    behaves exactly as before.
    """
    unique_months = sorted(dates.unique())
    
    train_months = train_months or len(unique_months) # way to set default if train_months is non-zero/NaN/Empty

    i = train_months
    while i + test_months <= len(unique_months):
        train_end = max(0, i - embargo_months)
        train_start = max(0, train_end - train_months)
        train_window = unique_months[train_start:train_end]
        test_window = unique_months[i:i + test_months]
        yield train_window, test_window
        i += step_months

In [33]:
walk_forward_split = walk_forward_splits(
        panel["date"], train_months=train_months, test_months=test_months,
        step_months=step_months, embargo_months=embargo_months
    )

In [54]:
for train_window, test_window in walk_forward_split:
    train_window = train_window
    test_window = test_window
    break

In [ ]:
train_window

[Timestamp('2012-11-30 00:00:00'),
 Timestamp('2012-12-31 00:00:00'),
 Timestamp('2013-01-31 00:00:00'),
 Timestamp('2013-02-28 00:00:00'),
 Timestamp('2013-03-31 00:00:00'),
 Timestamp('2013-04-30 00:00:00'),
 Timestamp('2013-05-31 00:00:00'),
 Timestamp('2013-06-30 00:00:00'),
 Timestamp('2013-07-31 00:00:00'),
 Timestamp('2013-08-31 00:00:00'),
 Timestamp('2013-09-30 00:00:00'),
 Timestamp('2013-10-31 00:00:00'),
 Timestamp('2013-11-30 00:00:00'),
 Timestamp('2013-12-31 00:00:00'),
 Timestamp('2014-01-31 00:00:00'),
 Timestamp('2014-02-28 00:00:00'),
 Timestamp('2014-03-31 00:00:00'),
 Timestamp('2014-04-30 00:00:00'),
 Timestamp('2014-05-31 00:00:00'),
 Timestamp('2014-06-30 00:00:00'),
 Timestamp('2014-07-31 00:00:00'),
 Timestamp('2014-08-31 00:00:00'),
 Timestamp('2014-09-30 00:00:00'),
 Timestamp('2014-10-31 00:00:00'),
 Timestamp('2014-11-30 00:00:00'),
 Timestamp('2014-12-31 00:00:00'),
 Timestamp('2015-01-31 00:00:00'),
 Timestamp('2015-02-28 00:00:00'),
 Timestamp('2015-03-

In [ ]:
test_window

[Timestamp('2018-01-31 00:00:00')]

In [ ]:
train = panel[panel["date"].isin(train_window)]
test = panel[panel["date"].isin(test_window)]

train = train.copy().sort_values("date")

# this groups all stocks by date... so for a data, it will have all the stocks
# so grouped will have 60 "rows"
grouped = train.groupby("date")[TARGET_COL]
for name, group in grouped:
    print(name)
    print(group)
    break

2012-11-30 00:00:00
22       0.070180
33862    0.155385
55715    0.183658
33679    0.185562
55958    0.102859
           ...   
44639    0.141044
46673    0.061649
46907    0.037609
46066    0.130436
45371    0.118729
Name: fwd_ret, Length: 476, dtype: float64


In [93]:
train[TARGET_COL] = train.groupby("date")[TARGET_COL].transform(winsorize)
X_train = train[FEATURE_COLS]
X_test, y_test = test[FEATURE_COLS], test[TARGET_COL]
X_train

,mom_1m_z,mom_3m_z,mom_12m_ex1_z,realized_vol_z,log_mkt_cap_z,log_dollar_vol_z
22,0.362149,-0.096395,-0.580769,-0.674850,2.396931,1.900974
33862,0.600791,0.041467,-0.732311,-0.080262,-0.116150,0.049656
55715,3.030582,0.814869,-2.550238,1.240456,-1.109893,-0.599655
33679,0.672277,-0.122245,-0.650063,0.716349,0.302468,0.650672
55958,1.596748,-0.222243,-0.843529,-0.117282,1.978803,2.194311
...,...,...,...,...,...,...
25111,0.996361,0.738902,0.005716,-1.036387,-0.483779,-1.046396
5462,-0.967899,-1.199261,-1.010940,-0.600441,0.514401,0.346403
65924,-1.830358,-1.540088,0.096828,1.202761,0.169806,0.847973
85448,-0.382210,-1.192409,-0.865280,0.125351,-0.180083,-1.187815


In [ ]:
X_test

,mom_1m_z,mom_3m_z,mom_12m_ex1_z,realized_vol_z,log_mkt_cap_z,log_dollar_vol_z
84,0.779718,-0.697439,0.109814,-0.548402,2.095194,1.672356
267,1.013047,0.413231,0.829293,-0.634615,2.724156,2.976923
451,0.299685,0.926663,1.945933,0.400530,0.036515,-0.127821
634,-0.046064,0.117129,0.694716,-0.759912,1.524652,1.172551
1002,0.417735,-0.339659,-1.059560,-0.163944,-0.094000,-0.316456
...,...,...,...,...,...,...
90804,0.454542,0.260403,0.955342,0.085139,2.167112,1.805257
91030,1.262150,2.044393,-1.824921,1.795081,-1.622251,-1.182523
91252,-1.198239,-1.305258,0.651510,0.719904,1.350250,1.809188
91378,-0.030204,0.654593,-0.031485,-1.382213,-0.470464,-1.093870


In [100]:
y_test

84      -0.111098
267     -0.011130
451      0.025796
634     -0.089476
1002     0.064785
           ...   
90804    0.001727
91030   -0.068732
91252   -0.068454
91378    0.063970
91538   -0.061071
Name: fwd_ret, Length: 476, dtype: float64

In [97]:
train # still the pure rows of the feature_panel, just sorted by date and only for the training window

,permno,cum_ret_index,mkt_cap,avg_dollar_vol,realized_vol,date,mom_1m,mom_3m,mom_12m_ex1,log_mkt_cap,log_dollar_vol,fwd_ret,mom_1m_z,mom_3m_z,mom_12m_ex1_z,realized_vol_z,log_mkt_cap_z,log_dollar_vol_z
22,10104,1.043996,1.527669e+11,5.892117e+08,0.011267,2012-11-30,0.035233,0.018585,-0.000356,25.752179,20.194296,0.070180,0.362149,-0.096395,-0.580769,-0.674850,2.396931,1.900974
33862,41355,0.987908,1.225522e+10,1.210372e+08,0.014969,2012-11-30,0.049896,0.032570,-0.036168,23.229218,18.611609,0.155385,0.600791,0.041467,-0.732311,-0.080262,-0.116150,0.049656
55715,75828,0.904147,4.519049e+09,6.947756e+07,0.023190,2012-11-30,0.199191,0.111027,-0.467444,22.231567,18.056514,0.183658,3.030582,0.814869,-2.550238,1.240456,-1.109893,-0.599655
33679,41080,0.920339,1.865688e+10,2.023310e+08,0.019928,2012-11-30,0.054288,0.015963,-0.016732,23.649481,19.125416,0.185562,0.672277,-0.122245,-0.650063,0.716349,0.302468,0.650672
55958,76076,0.970543,1.003979e+11,7.571478e+08,0.014738,2012-11-30,0.111091,0.005818,-0.062451,25.332407,20.445069,0.102859,1.596748,-0.222243,-0.843529,-0.117282,1.978803,2.194311
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25111,24985,2.933550,1.504094e+10,7.008925e+07,0.005584,2017-10-31,0.071750,0.113167,0.195871,23.434042,18.065280,-0.079658,0.996361,0.738902,0.005716,-1.036387,-0.483779,-1.046396
5462,12558,0.699789,4.044398e+10,2.217855e+08,0.008395,2017-10-31,-0.049267,-0.107440,-0.044322,24.423184,19.217221,-0.000372,-0.967899,-1.199261,-1.010940,-0.600441,0.514401,0.346403
65924,81061,2.056565,2.874440e+10,3.358095e+08,0.020024,2017-10-31,-0.102403,-0.146234,0.217397,24.081709,19.632055,0.227649,-1.830358,-1.540088,0.096828,1.202761,0.169806,0.847973
85448,90442,0.987916,2.032236e+10,6.235267e+07,0.013076,2017-10-31,-0.013183,-0.106660,-0.009908,23.734988,17.948317,0.432101,-0.382210,-1.192409,-0.865280,0.125351,-0.180083,-1.187815


In [105]:
train_group = train.groupby("date").size().to_numpy()
train_group

array([476, 480, 481, 481, 480, 478, 479, 481, 482, 481, 478, 479, 478,
       480, 480, 478, 477, 475, 477, 478, 478, 478, 481, 482, 479, 484,
       479, 478, 479, 478, 476, 476, 477, 478, 477, 476, 473, 472, 470,
       467, 468, 466, 466, 467, 468, 469, 472, 471, 473, 472, 469, 470,
       474, 474, 474, 472, 474, 473, 476, 476])

In [ ]:
rank_label = train.groupby("date")[TARGET_COL].transform(lambda s: pd.qcut(s, 10, labels=False, duplicates="drop"))
rank_label